# Repairing the signature grid's baseline - zero GPU hours

`ablation_signature_only.ipynb` is the grid behind the study's sharpest attribution result: that
**76% (indication) and 93% (contraindication)** of `disease_protein`'s effect is signature-side rather
than message-passing-side. It has one defect, and it is a bookkeeping defect rather than a scientific
one.

### The defect

Its four ablation arms ran seeds **10-19**. Its `baseline` arm ran seeds **{1, 2, 3, 4, 5, 10}**. The
notebook's aggregation pairs each arm against the baseline on an inner join over seed, and the two sets
intersect in exactly **one** seed. So `results/ablation_signature_only/signature_only_summary.csv`
reports every delta from a single pair: `indication_n_paired = 1.0`, every confidence interval `NaN`.

The published figures in `docs/ABLATION_FINDINGS.md` are **not** affected - they were computed by
pairing those arms against `ablation_stage_a_seeds10_19/baseline` instead, after verifying that grid is
configuration-identical. But the committed CSV in the repo disagrees with the write-up, which is the
kind of discrepancy that costs an afternoon in six months' time.

### The fix, and why it is free

`ablation_stage_a_seeds10_19/baseline` has seeds 10-19 at **exactly** this configuration: same
`model_config` (all 11 keys), same `pretrain`, same `finetune`, same split, `dropped_edges_total = 0`,
same `signature_disease_etypes`. Nine of those runs - seeds 11-19 - are precisely the missing baselines.

So this notebook **imports** them rather than training them. 9 runs, ~4 GPU-hours saved, and the result
is not an approximation: a baseline run is fully determined by (split seed, config, schedule), all of
which are verified field by field before any file is copied. Section 2 refuses the import and tells you
which field disagreed if any of them does.

It then re-aggregates the grid with correct pairing and writes a **new** file,
`signature_only_summary_paired.csv`. The original committed CSV is left untouched - this notebook never
overwrites a result another notebook produced.

---
## 1 - Environment

No torch, no CUDA, no model: this notebook only reads and copies result files.

In [ ]:
import os, sys, json, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()

SIG_ROOT    = REPO_ROOT / 'results' / 'ablation_signature_only'
SOURCE_ROOT = REPO_ROOT / 'results' / 'ablation_stage_a_seeds10_19'
SOURCE_ARM  = 'baseline'
TARGET_ARM  = 'baseline'

SPLIT      = 'complex_disease'
WANT_SEEDS = [11, 12, 13, 14, 15, 16, 17, 18, 19]   # the gap; seed 10 is already there
ARM_SEEDS  = list(range(10, 20))                    # what the four ablation arms actually ran

DRY_RUN = False   # True lists what would be imported without writing anything

for p in (SIG_ROOT, SOURCE_ROOT):
    assert p.is_dir(), 'missing %s' % p

print('repo   :', REPO_ROOT)
print('target :', SIG_ROOT.relative_to(REPO_ROOT))
print('source :', SOURCE_ROOT.relative_to(REPO_ROOT))


def seeds_of(root, arm):
    out = []
    for d in sorted(Path(root).glob('%s_seed*' % arm)):
        if (d / 'run_meta.json').exists() and (d / 'edge_level_test_metrics.csv').exists():
            out.append(int(d.name.rsplit('seed', 1)[1]))
    return sorted(out)


print('\narms and seeds currently on disk:')
for arm in sorted({d.name.rsplit('_seed', 1)[0] for d in SIG_ROOT.iterdir()
                   if d.is_dir() and '_seed' in d.name}):
    print('  %-26s %s' % (arm, seeds_of(SIG_ROOT, arm)))
print('\nsource %s/%s: %s' % (SOURCE_ROOT.name, SOURCE_ARM, seeds_of(SOURCE_ROOT, SOURCE_ARM)))

---
## 2 - Verify, then import

Every field below must match before a file is copied. `model_config` is compared on **all** of its keys,
not a subset, because a baseline run differing in `exp_lambda` or `proto_num` would not be a valid
reference for these arms.

In [ ]:
# the full config that defines a baseline run of this grid, read from a run that is already here
_ref_meta = json.loads((SIG_ROOT / ('%s_seed10' % TARGET_ARM) / 'run_meta.json').read_text())
EXPECT = {
    'split': SPLIT,
    'dropped_edges_total': 0,
    'signature_disease_etypes': _ref_meta['signature_disease_etypes'],
    'model_config': _ref_meta['model_config'],      # all 11 keys, including proto
    'pretrain': _ref_meta['pretrain'],
    'finetune': _ref_meta['finetune'],
}
# NOT a top-level 'proto' key: the replication grid predates that field and records metric learning
# only inside model_config. model_config is compared on every key, so proto is still enforced - this
# asserts that explicitly rather than leaving it implicit.
assert EXPECT['model_config']['proto'] is True, 'the reference run did not use metric learning'
print('reference configuration taken from %s_seed10:' % TARGET_ARM)
for k, v in EXPECT.items():
    print('  %-26s %s' % (k, v))


def check(meta, seed):
    '''Return a list of human-readable mismatches against EXPECT (empty == identical).'''
    bad = []
    if meta.get('seed') != seed:
        bad.append('seed: %r != %r' % (meta.get('seed'), seed))
    for k, want in EXPECT.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                bad.append('%s is not a dict' % k)
                continue
            if set(got) != set(want):
                bad.append('%s keys differ: %s' % (k, set(got) ^ set(want)))
            for kk, vv in want.items():
                if got.get(kk, '<missing>') != vv:
                    bad.append('%s[%s]: %r != %r' % (k, kk, got.get(kk, '<missing>'), vv))
        elif got != want:
            bad.append('%s: %r != %r' % (k, got, want))
    return bad


rows = []
for seed in WANT_SEEDS:
    src = SOURCE_ROOT / ('%s_seed%d' % (SOURCE_ARM, seed))
    dst = SIG_ROOT / ('%s_seed%d' % (TARGET_ARM, seed))
    if dst.exists() and (dst / 'run_meta.json').exists():
        rows.append({'seed': seed, 'status': 'already present', 'detail': ''})
        continue
    if not (src / 'run_meta.json').exists():
        rows.append({'seed': seed, 'status': 'NO SOURCE', 'detail': str(src.name)})
        continue

    meta = json.loads((src / 'run_meta.json').read_text())
    bad = check(meta, seed)
    if bad:
        rows.append({'seed': seed, 'status': 'REFUSED', 'detail': '; '.join(bad[:3])})
        continue
    if DRY_RUN:
        rows.append({'seed': seed, 'status': 'would import', 'detail': src.name})
        continue

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src / 'edge_level_test_metrics.csv', dst / 'edge_level_test_metrics.csv')
    meta['run_key'] = dst.name
    meta['arm'] = TARGET_ARM
    meta['experiment'] = 'signature_only'
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['imported_note'] = ('baseline gap fill: this grid trained baselines only on seeds '
                             '{1-5,10} while its ablation arms ran 10-19')
    (dst / 'run_meta.json').write_text(json.dumps(meta, indent=2, default=str))
    rows.append({'seed': seed, 'status': 'imported', 'detail': str(src.relative_to(REPO_ROOT))})

display(pd.DataFrame(rows).style.hide(axis='index'))

now = seeds_of(SIG_ROOT, TARGET_ARM)
print('\n%s seeds now: %s' % (TARGET_ARM, now))
missing = sorted(set(ARM_SEEDS) - set(now))
if missing:
    print('[warn] still missing baselines for arm seeds: %s' % missing)
else:
    print('[ok] every seed the ablation arms used (%s) now has a baseline.' % ARM_SEEDS)

---
## 3 - Re-aggregate with correct pairing

Writes `signature_only_summary_paired.csv` next to the original. The original
`signature_only_summary.csv` is **not** modified.

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'sig_dim': meta.get('signature_dim'),
                         'sig_max': meta.get('signature_max'),
                         'proto': meta.get('proto'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}
REF_ARM = 'baseline'
ARM_ORDER = ['baseline', 'sig_drop_disease_protein', 'sig_drop_disease_disease',
             'sig_drop_both', 'proto_off']

long = _load_all(SIG_ROOT)
wide = (long[long.relation.isin(ROW.values())]
        .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
        .rename(columns={v: k for k, v in ROW.items()}))
ref = wide.xs(REF_ARM, level='arm')

out = []
for arm, g in wide.groupby(level='arm'):
    g = g.droplevel('arm')
    rec = {'arm': arm, 'n_seeds': len(g), 'seeds': ','.join(str(s) for s in sorted(g.index))}
    for key in ROW:
        rec['%s_AUPRC' % key] = g[key].mean()
        rec['%s_CI' % key] = _ci95(g[key])
        if arm == REF_ARM:
            rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
            rec['%s_n_paired' % key] = len(g)
        else:
            paired = (g[key] - ref[key]).dropna()      # inner join on seed
            rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
            rec['%s_delta_CI' % key] = _ci95(paired)
            rec['%s_n_paired' % key] = len(paired)
            rec['%s_hurt' % key] = int((paired < 0).sum())
    out.append(rec)

summary = (pd.DataFrame(out)
             .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(ARM_ORDER)}))
             .sort_values(['_o', 'arm']).drop(columns='_o').reset_index(drop=True))
summary.to_csv(SIG_ROOT / 'signature_only_summary_paired.csv', index=False)

cols = [c for c in ['arm', 'n_seeds', 'indication_delta', 'indication_delta_CI',
                    'indication_n_paired', 'indication_hurt', 'contraindication_delta',
                    'contraindication_delta_CI', 'contraindication_n_paired',
                    'contraindication_hurt', 'micro_delta', 'micro_delta_CI']
        if c in summary.columns]
print('-> %s\n' % (SIG_ROOT / 'signature_only_summary_paired.csv').relative_to(REPO_ROOT))
display(summary[cols].style.format('{:.4f}', subset=[
    c for c in cols if c.endswith(('delta', 'CI'))], na_rep='-').hide(axis='index'))

old_p = SIG_ROOT / 'signature_only_summary.csv'
if old_p.exists():
    old = pd.read_csv(old_p)
    if 'indication_n_paired' in old.columns:
        print('\nbefore / after, paired-seed counts:')
        comp = (old[['arm', 'indication_n_paired']]
                .merge(summary[['arm', 'indication_n_paired']], on='arm',
                       suffixes=(' (committed)', ' (repaired)')))
        display(comp.style.hide(axis='index'))

### 3.1 - The decomposition, recomputed

The result this grid exists for: splitting each relation's effect into a signature-side component
(measured here, graph untouched) and a message-passing residual (the leave-one-out effect from
`ablation_stage_a_seeds10_19` minus this). These should reproduce section 6 of
`docs/ABLATION_FINDINGS.md`: shares of **76% / 93% / 86%** for `disease_protein`.

In [ ]:
REPL_ROOT = REPO_ROOT / 'results' / 'ablation_stage_a_seeds10_19'

# graph+signature effects from the replication grid, paired on the seeds THIS grid used
repl = _load_all(REPL_ROOT)
rows = []
if repl.empty:
    print('%s not found - cannot compute residuals.' % REPL_ROOT.name)
else:
    rwide = (repl[repl.relation.isin(ROW.values())]
             .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
             .rename(columns={v: k for k, v in ROW.items()}))
    rref = rwide.xs('baseline', level='arm')

    PAIRS = [('disease_protein', 'drop_disease_protein_full', 'sig_drop_disease_protein'),
             ('disease_disease', 'drop_disease_disease_full', 'sig_drop_disease_disease')]
    for rel, loo_arm, sig_arm in PAIRS:
        if loo_arm not in set(rwide.index.get_level_values('arm')):
            continue
        if sig_arm not in set(wide.index.get_level_values('arm')):
            continue
        sig_seeds = sorted(wide.xs(sig_arm, level='arm').index)
        for key in ROW:
            # total: full-KG leave-one-out, restricted to the seeds this grid used
            g_loo = rwide.xs(loo_arm, level='arm')[key]
            total = (g_loo - rref[key]).dropna()
            total = total[total.index.isin(sig_seeds)]
            sig = (wide.xs(sig_arm, level='arm')[key] - ref[key]).dropna()
            common = sorted(set(total.index) & set(sig.index))
            if not common:
                continue
            t = total.loc[common]
            s = sig.loc[common]
            resid = t - s
            rows.append({'relation': rel, 'metric': key, 'n': len(common),
                         'total (graph+sig)': t.mean(), 'signature only': s.mean(),
                         'residual (msg passing)': resid.mean(),
                         'residual CI95': _ci95(resid),
                         'signature share %': (100 * s.mean() / t.mean()) if t.mean() else np.nan})

    if rows:
        dec = pd.DataFrame(rows)
        dec.to_csv(SIG_ROOT / 'signature_decomposition_paired.csv', index=False)
        display(dec.style.format('{:.4f}', subset=[
            'total (graph+sig)', 'signature only', 'residual (msg passing)', 'residual CI95'],
            na_rep='-').format({'signature share %': '{:.0f}'}, na_rep='-').hide(axis='index'))
        print('-> %s' % (SIG_ROOT / 'signature_decomposition_paired.csv').relative_to(REPO_ROOT))
        print('\nA residual whose CI spans zero means the relation\'s edges can stay in the graph as')
        print('long as the signature cannot read them - the central claim of the comment paper.')
    else:
        print('No overlapping arms/seeds found to decompose.')

---
## 4 - After running this

1. `results/ablation_signature_only/` now has a baseline for every seed its arms used, so the grid is
   self-contained and no longer depends on a cross-grid reference.
2. `signature_only_summary_paired.csv` and `signature_decomposition_paired.csv` are the files to cite.
   The committed `signature_only_summary.csv` is unchanged and should be regarded as superseded - it is
   the one with `n_paired = 1`.
3. The housekeeping item in `docs/ABLATION_FINDINGS.md` section 12 and the pre-submission checklist item
   in the comment-paper draft can both be struck, and the manuscript's note that the signature figures
   are paired cross-grid can be simplified to within-grid pairing.
4. Nothing here changes a reported number. If section 3.1 does **not** reproduce 76% / 93% / 86% for
   `disease_protein`, that is a real discrepancy worth chasing before publication - the figures in the
   write-up were computed by exactly this pairing, just from a script rather than from this notebook.